In [5]:
import openai
import os
from qdrant_client import QdrantClient
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langsmith import Client
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper


In [6]:
from dotenv import load_dotenv
import os

load_dotenv("../../.env")

True

###  Download example eval dataset from Langsmith



In [7]:
ls_client = Client()


In [8]:
dataset = ls_client.read_dataset(dataset_name="rag-eval-dataset")

In [9]:
list(ls_client.list_examples(dataset=dataset.id, limit=50))[12].outputs

{'ground_truth': 'The following audio products feature an IPX7 waterproof rating: 1) Raymate Bluetooth Speakers (B0C996WY16) with 30W HiFi sound and 1000mins playtime. 2) Wekily Bluetooth 5.3 Wireless Earbuds (B0BRV544MV) with running earhooks and 40H total playtime. 3) Jesebang Wireless Earbuds (B09X9838WY) with 13mm drivers and 40H playtime.',
 'reference_context_ids': ['B0C996WY16', 'B0BRV544MV', 'B09X9838WY'],
 'reference_descriptions': ['Raymate Bluetooth Speakers, HiFi Stereo Sound with DSP, 30W IPX7 Waterproof Speaker Wireless Bluetooth-V5.0, 1000mins Playtime, Portable Speaker for Home, Outdoor, Party 🎶HiFi Sound: With proven audio processing DSP chip technology, 30W dual speaker drivers and a more powerful amplifier module, the portable speakers delivers even, Balanced Sound Without Distortion. 🧱Integrated structure: With IPX7 Waterproof Speakers protection against rain, dust, snow and splashes, you can enjoy music in the pool, beach, park, bathroom and beyond.Comes with a dir

In [12]:
list(ls_client.list_examples(dataset=dataset.id, limit=50))[12].inputs

{'question': 'Which of your wireless audio products feature an IPX7 waterproof rating for workout or outdoor use?'}

In [13]:
list(ls_client.list_examples(dataset=dataset.id, limit=50))[0].outputs


{'ground_truth': 'Currently, we do not have standalone PlayStation 5 consoles available for purchase. We do, however, sell products optimized for PS5 gaming, such as the Sony 85 Inch 4K Ultra HD TV X90K Series.',
 'reference_context_ids': [],
 'reference_descriptions': []}

In [14]:
reference_input = list(ls_client.list_examples(dataset=dataset.id, limit=50))[12].inputs
reference_output = list(ls_client.list_examples(dataset=dataset.id, limit=50))[12].outputs

### RAG Pipeline

In [ ]:
from openai import OpenAI
import os
from qdrant_client import QdrantClient
from langsmith import traceable, get_current_run_tree
from groq import Groq

groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))
#from langsmith import traceable, get_current_run_tree

qdrant_client = QdrantClient(url='http://qdrant:6333')

client = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)


def get_embedding(text, model="gemini-embedding-001"):
    '''Gemini embedding has no usage tracking '''
    response = client.embeddings.create(
        model=model,
        input=text,
    )
    # current_run = get_current_run_tree()
    # if current_run:
    #     current_run.metadata['usage_metadata'] = {
    #         'input_tokens': response.usage.prompt_tokens,
    #         'total_tokens': response.usage.total_tokens,
    #     }
    return response.data[0].embedding



def retrieve_data(query, collection_name='amazon-items-collection-00', k=5):
    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name=collection_name,
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context_scores = []
    retrieved_context_texts = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload['parent_asin'])
        retrieved_context_scores.append(result.score)
        retrieved_context_texts.append(result.payload['processed_description'])
        retrieved_context_ratings.append(result.payload['average_rating'])

    return {
        'retrieved_context_ids': retrieved_context_ids,
        'retrieved_context_scores': retrieved_context_scores,
        'retrieved_context_texts': retrieved_context_texts,
        'retrieved_context_ratings': retrieved_context_ratings
    }


def process_context(retrieve_context):
    formatted_context = ''

    for id, chunk, rating in zip(retrieve_context['retrieved_context_ids'], retrieve_context['retrieved_context_texts'], retrieve_context['retrieved_context_ratings']):
        formatted_context += f"- Product ID: {id}, Product Rating: {rating}, Product Description: {chunk}\n"

    return formatted_context


def build_prompt(question, formatted_context):
    prompt = f"""
    You are a shopping assistant that can answer questions about the products in stock.

    You will be given a question and a list of context.

    Instructions:
    - Answer the question based on the context only.
    - Never use word context and refer to it as the available products.
    - Do not use markdown formatting

    Context:
    {formatted_context}

    Question:
    {question}
    """
    return prompt



def generate_answer(prompt):
    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {"role": "user", "content": prompt}
        ],
        reasoning_effort="low",
        max_completion_tokens=1000,
    )

    return response.choices[0].message.content


def rag_pipeline(question, topk=5):
    retrieve_context = retrieve_data(query=question, k=topk)
    formatted_context = process_context(retrieve_context)
    prompt = build_prompt(question=question, formatted_context=formatted_context)
    answer = generate_answer(prompt)

    return {
        'answer': answer,
        'question': question,
        'retrieved_context_ids': retrieve_context['retrieved_context_ids'],  
        'retrieved_context': retrieve_context['retrieved_context_texts'],    
    }  